# **NLP Medical Question Filtering**


***
# 0. Setup

## Imports & Utilities

In [1]:
# General imports
import os
from datetime import datetime
import time

import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import uuid

# Plots
import matplotlib.pyplot as plt
plt.style.use('ggplot')

# Embedding
from sentence_transformers import SentenceTransformer

# Scikit Learn
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

# Tensorflow, Keras
import tensorflow as tf 
import keras_tuner as kt


In [2]:
# Utility for dataset summary
#

def display_file_summary(data_frame, name=""):
    # Create a temporary df and ensure no lists remain, so that unique items can be identified for uniqueness
    temp_df = data_frame.copy()
    temp_df = temp_df.map(lambda cell: str(cell) if isinstance(cell, list) else cell)
    
    # Calculate Data Types 
    summary_of_df = pd.DataFrame({'Count': data_frame.count(),
                                 'Missing': data_frame.isnull().sum(), 'Empty': 0,
                                 'Unique': temp_df.nunique(),
                                 'Top': data_frame.apply(lambda x: x.value_counts().index[0]),
                                 'Freq': data_frame.apply(lambda x: x.value_counts().iat[0]),
                                 'Type': data_frame.dtypes, 
                                 'String': 0, 'Int': 0, 'Float': 0, 'Bool': 0, 'List': 0, 'Other': 0
                                 })
    # ?? Below with Map should not work as counting entire data frame, but counts do appear correct
    #test = data_frame['Description'].apply(lambda cell: isinstance(cell, str)).sum()
    summary_of_df['Empty'] = (data_frame == '').sum()
    summary_of_df['String'] = data_frame.map(lambda cell: isinstance(cell, str)).sum()
    summary_of_df['Int'] = data_frame.map(lambda cell: isinstance(cell, int)).sum()
    summary_of_df['Float'] = data_frame.map(lambda cell: isinstance(cell, float)).sum()
    summary_of_df['Bool'] = data_frame.map(lambda cell: isinstance(cell, bool)).sum()
    summary_of_df['List'] = data_frame.map(lambda cell: isinstance(cell, list)).sum()
    summary_of_df['Other'] = data_frame.map(lambda cell: isinstance(cell, (tuple,dict,set,type(None)))).sum()

    print(f'File Details. {name}')
    display(summary_of_df)

## Project Setup

In [3]:
# Project Setup
#

#---- Run Parameters --------------------------------
run_name = 'Test_Training_v1'

#----------------------------------------------------

# Folder paths
local_project_folder = Path.cwd().parent
data_folder = local_project_folder.joinpath('Data')
if not data_folder.exists():
    raise FileNotFoundError(f'{data_folder} does not exist')
run_results_folder = local_project_folder.joinpath(f'Results/{run_name}')
if not run_results_folder.exists():
    raise FileNotFoundError(f'{run_results_folder} does not exist')

del local_project_folder, data_folder, run_name

***
# Sentence Embedding

## Functions

In [4]:
# Sentence embedding
#

def create_embeddings(sentences):
    # TODO: Understand why vector allignment is needed and how it is performed, see Antonio
    # See https://www.kaggle.com/discussions/general/566301

    # SBERT with all-MiniLM-L6-V2, a small and fast model, 385 dimensions
    # ANTONIO: encoding_models = {'all-MiniLM-L6-v2': 'sbert22M'}
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = sbert_model.encode(sentences, convert_to_numpy=True, show_progress_bar=True)

    # Allign vectors to the axis and rotate
    u, s, vh = np.linalg.svd(a=embeddings)
    align_mat = np.linalg.solve(a=vh, b=np.eye(len(embeddings[0])))
    embeddings = np.matmul(embeddings, align_mat)
    print(f"Embeddings shape: {embeddings.shape}")
    # query_df['embeddings'] = list(embeddings)

    return embeddings

## Execute

In [5]:
# Load the raw dataset and split for training

# TODO: Have not split neg and pos, ANTONIO has though????
# TODO: Spliting X,y AFTER embeddings

# Get the raw training dataset
query_df = pd.read_pickle(run_results_folder.joinpath('query_data_raw.pkl'))

# Get the raw training dataset & add embeddings
query_df = pd.read_pickle(run_results_folder.joinpath('query_data_raw.pkl'))
query_df['query-embedding'] = list(create_embeddings(query_df['query'].values))
print(f"Query Embeddings df shape: {query_df.shape}")

# Save
query_df.to_pickle(run_results_folder.joinpath('query_data_embeddings.pkl'))

Batches:   0%|          | 0/92 [00:00<?, ?it/s]

Embeddings shape: (2916, 384)
Query Embeddings df shape: (2916, 5)


# Temp# Create a DataFrame with test queries and predictionstest_results_df = pd.DataFrame({    'query': X_text_test,    'actual_label': y_test,    'actual_safe': ["Safe" if label == 1 else "Unsafe" for label in y_test],    'predicted_prob': y_pred_probs.flatten(),    'predicted_binary': (y_pred_probs > 0.5).astype(int).flatten(),    'predicted_safe': ["Safe" if prob > 0.5 else "Unsafe" for prob in y_pred_probs.flatten()],    'correct_prediction': (y_test == (y_pred_probs > 0.5).astype(int).flatten())})print(f"Test results DataFrame shape: {test_results_df.shape}")print("\nFirst 5 rows:")display(test_results_df.head())

# Inspect queries and predictions - on the test dataset split
print(f"Correct predictions: {test_results_df['correct_prediction'].sum()}/{len(test_results_df)} ({test_results_df['correct_prediction'].mean():.3f})")

# Quick look at queries and predictions
for i in range(min(15, len(y_pred_probs))):
    query = X_text_test[i]
    actual_flag = "Safe" if y_test[i] == 1 else "Unsafe"
    predicted_prob = y_pred_probs[i][0]

# Predicted probabilities
y_pred_probs = simple_model.predict(X_test)

# Create a DataFrame with test queries and predictions
test_results_df = pd.DataFrame({    'query': X_text_test,    'actual_label': y_test,    'actual_safe': ["Safe" if label == 1 else "Unsafe" for label in y_test],    'predicted_prob': y_pred_probs.flatten(),    'predicted_binary': (y_pred_probs > 0.5).astype(int).flatten(),    'predicted_safe': ["Safe" if prob > 0.5 else "Unsafe" for prob in y_pred_probs.flatten()],    'correct_prediction': (y_test == (y_pred_probs > 0.5).astype(int).flatten())})print(f"Test results DataFrame shape: {test_results_df.shape}")print("\nFirst 5 rows:")
display(test_results_df.head())

print("\nPrediction accuracy summary:")
print(f"Correct predictions: {test_results_df['correct_prediction'].sum()}/{len(test_results_df)} ({test_results_df['correct_prediction'].mean():.3f})")

# Save the results DataFrame
test_results_df.to_pickle(data_folder.joinpath('test_results_with_predictions.pkl'))
print("\nSaved test results to 'test_results_with_predictions.pkl'")

# Quick look at queries and predictions
for i in range(min(15, len(y_pred_probs))):
    query = X_text_test[i]
    actual_flag = "Safe" if y_test[i] == 1 else "Unsafe"
    predicted_prob = y_pred_probs[i][0]

    print(f"{query}")
    print(f"- Actual: {actual_flag} vs Predicted: {predicted_prob:.2f}")

#
